# 01 - Convergence, Energetics and Materials Taxonomy

This notebook asks whether the database is numerically trustworthy enough for
materials-science interpretation. The key idea is simple:

- a chemically rich database is only useful if the calculations are converged,
- but convergence should be analyzed by material class, not only globally.

The `DFTDataFrame` package already contains the relevant helper functions:
`OP.converged(...)` and `OP.notconverged(...)`.

In [ ]:
from pathlib import Path
from collections import Counter
import math
import re
import subprocess
import sys
import tempfile

sys.path.insert(0, r"/Users/dk2994/Desktop/git/DFTDataFrame/src")
import DFTDataFrame.Tools as OP

pd = OP.pd
np = OP.np
plt = OP.plt

DATA_PATH = Path(r"/Users/dk2994/Desktop/Uni/scripts/created_frame.hdf")
DFTDATAFRAME_SRC = Path(r"/Users/dk2994/Desktop/git/DFTDataFrame/src")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12


def load_onepiece_hdf(path=DATA_PATH, key="df"):
    """Load the local OnePiece-style HDF table.

    This notebook uses the local DFTDataFrame package as the available
    OnePiece-compatible analysis layer. The actual HDF payload is read through
    the pandas namespace exposed by that package.
    """
    path = Path(path)
    try:
        return OP.pd.read_hdf(path, key=key).copy()
    except Exception as original_error:
        helper_python = Path("/Users/dk2994/opt/anaconda3/bin/python")
        if not helper_python.exists():
            raise original_error
        output = Path(tempfile.NamedTemporaryFile(delete=False, suffix=".pkl", prefix="created_frame_").name)
        script = """
from pathlib import Path
import sys
import numpy as np
import pandas as pd
try:
    import numpy.core as numpy_core
    sys.modules.setdefault("numpy._core", numpy_core)
    sys.modules.setdefault("numpy._core.multiarray", np.core.multiarray)
    sys.modules.setdefault("numpy._core.numeric", np.core.numeric)
    import ase.constraints  # noqa: F401
except Exception:
    pass
source = Path(sys.argv[1])
key = sys.argv[2]
target = Path(sys.argv[3])
pd.read_hdf(source, key=key).to_pickle(target)
"""
        completed = subprocess.run(
            [str(helper_python), "-c", script, str(path), key, str(output)],
            capture_output=True,
            text=True,
            check=False,
        )
        if completed.returncode != 0:
            detail = completed.stderr.strip() or completed.stdout.strip()
            raise RuntimeError(f"Could not read {path}. Helper reader failed: {detail}") from original_error
        return OP.pd.read_pickle(output).copy()


ADSORBATE_TOKENS = [
    "CH3OH", "CH3O", "HCOOH", "H2COOH", "HCOO", "COOH", "CO2", "HCO", "CO"
]


def guess_adsorbate(name):
    if not isinstance(name, str):
        return ""
    for token in sorted(ADSORBATE_TOKENS, key=len, reverse=True):
        if re.search(rf"(^|[-_%]){re.escape(token)}($|[-_%])", name):
            return token
    return ""


def guess_record_class(name, path=""):
    text = f"{name} {path}".lower()
    if "gasphases" in text:
        return "gas"
    if "copt" in text:
        return "copt"
    if "convergence" in text:
        return "convergence"
    if "bulk" in text:
        return "bulk"
    if "clean" in text:
        return "clean_surface"
    if guess_adsorbate(name):
        return "adsorbate"
    return "other"


def guess_facet(name):
    if not isinstance(name, str):
        return ""
    for facet in ["100", "110", "111", "211", "221"]:
        if facet in name:
            return facet
    return ""


def material_family(name):
    if not isinstance(name, str):
        return "unknown"
    token = name.split("-")[0]
    return token or "unknown"


def surface_key_from_name(name):
    if not isinstance(name, str):
        return ""
    key = name
    key = re.sub(r"-copt-.*$", "", key)
    key = re.sub(r"-(CH3OH|CH3O|HCOOH|H2COOH|HCOO|COOH|CO2|HCO|CO)([-_%].*)?$", "", key)
    key = re.sub(r"-[0-9]+$", "", key)
    return key


def add_taxonomy(df):
    out = df.copy()
    out["record_class"] = [guess_record_class(n, p) for n, p in zip(out["Name"], out["Path"], strict=False)]
    out["facet"] = out["Name"].map(guess_facet)
    out["material_family"] = out["Name"].map(material_family)
    out["adsorbate"] = out["Name"].map(guess_adsorbate)
    out["surface_key"] = out["Name"].map(surface_key_from_name)
    return out


def formula_counts(formula):
    counts = {}
    if not isinstance(formula, str):
        return counts
    for element, number in re.findall(r"([A-Z][a-z]?)(\d*)", formula):
        counts[element] = counts.get(element, 0) + int(number or 1)
    return counts


def add_formula_features(df):
    out = df.copy()
    parsed = out["Formula"].map(formula_counts)
    all_elements = sorted({el for counts in parsed for el in counts})
    for el in all_elements:
        out[el] = parsed.map(lambda counts: counts.get(el, 0))
    out["n_elements"] = parsed.map(len)
    out["n_atoms_formula"] = parsed.map(lambda counts: sum(counts.values()))
    return out


def gas_reference_energy(df, token):
    pattern = rf"^gasphases-{re.escape(token)}(?:$|[-_].*)"
    subset = df[df["Name"].astype(str).str.contains(pattern, regex=True, case=False, na=False)]
    subset = subset[pd.to_numeric(subset["E"], errors="coerce").notna()]
    if subset.empty:
        return np.nan
    return float(subset["E"].iloc[0])


def assign_clean_references(df):
    out = df.copy()
    clean = out[out["record_class"] == "clean_surface"].copy()
    clean = clean[pd.to_numeric(clean["E"], errors="coerce").notna()]
    clean_map = clean.groupby("surface_key")["E"].min().to_dict()
    clean_name_map = clean.sort_values("E").drop_duplicates("surface_key").set_index("surface_key")["Name"].to_dict()
    out["clean_ref_E"] = out["surface_key"].map(clean_map)
    out["clean_ref_name"] = out["surface_key"].map(clean_name_map)
    out["delta_E_surface"] = pd.to_numeric(out["E"], errors="coerce") - pd.to_numeric(out["clean_ref_E"], errors="coerce")
    return out


def adsorption_energy_conventions(df):
    out = df.copy()
    e_co = gas_reference_energy(out, "CO")
    e_h2 = gas_reference_energy(out, "H2")
    e_ch3oh = gas_reference_energy(out, "CH3OH")
    e_hcooh = gas_reference_energy(out, "HCOOH")
    out["E_ads_CO"] = np.where(out["adsorbate"].eq("CO"), out["E"] - out["clean_ref_E"] - e_co, np.nan)
    out["E_ads_CH3O_from_CH3OH"] = np.where(
        out["adsorbate"].eq("CH3O"),
        out["E"] - out["clean_ref_E"] - e_ch3oh + 0.5 * e_h2,
        np.nan,
    )
    out["E_ads_HCOO_from_HCOOH"] = np.where(
        out["adsorbate"].eq("HCOO"),
        out["E"] - out["clean_ref_E"] - e_hcooh + 0.5 * e_h2,
        np.nan,
    )
    out["E_ads_COOH_from_HCOOH"] = np.where(
        out["adsorbate"].eq("COOH"),
        out["E"] - out["clean_ref_E"] - e_hcooh + 0.5 * e_h2,
        np.nan,
    )
    return out


df_raw = load_onepiece_hdf()
df = add_formula_features(add_taxonomy(df_raw))
df["E"] = pd.to_numeric(df["E"], errors="coerce")
df["fmax"] = pd.to_numeric(df["fmax"], errors="coerce")
df["has_structure"] = df["struc"].map(lambda value: value.__class__.__name__ == "Atoms")
df["has_contcar"] = df["CONTCAR"].map(lambda value: value.__class__.__name__ == "Atoms")

## 1. Split converged and non-converged calculations

In [ ]:
df_nonzero = df[df["E"].notna() & (df["E"] != 0)].copy()
df_converged = OP.converged(df_nonzero, force_col="fmax", convergence_threshold=0.01)
df_notconverged = OP.notconverged(df_nonzero, force_col="fmax", convergence_threshold=0.01)

summary = pd.DataFrame({
    "subset": ["all nonzero", "converged", "not converged"],
    "rows": [len(df_nonzero), len(df_converged), len(df_notconverged)],
    "fraction": [1.0, len(df_converged) / len(df_nonzero), len(df_notconverged) / len(df_nonzero)],
})
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df_nonzero["fmax"], bins=120, color="#4c78a8", alpha=0.75)
ax.set_yscale("log")
ax.axvline(0.01, color="crimson", linestyle="--", label="convergence threshold = 0.01 eV/Å")
ax.set_title("Distribution of fmax for nonzero-energy calculations")
ax.set_xlabel("fmax / eV Å$^{-1}$")
ax.set_ylabel("count (log scale)")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Which record classes carry the convergence burden?

In [ ]:
class_conv = (
    df_nonzero.groupby("record_class")
    .agg(rows=("Name", "size"), mean_fmax=("fmax", "mean"), median_fmax=("fmax", "median"))
    .sort_values("rows", ascending=False)
)
class_conv["fraction_converged"] = [
    OP.converged(group, force_col="fmax", convergence_threshold=0.01).shape[0] / len(group)
    for _, group in df_nonzero.groupby("record_class")
]
class_conv

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
class_conv["fraction_converged"].sort_values().plot(kind="barh", ax=axes[0], color="#72b7b2")
axes[0].set_title("Fraction converged by record class")
axes[0].set_xlabel("fraction with fmax < 0.01")
class_conv["median_fmax"].sort_values().plot(kind="barh", ax=axes[1], color="#e45756")
axes[1].set_title("Median fmax by record class")
axes[1].set_xlabel("median fmax / eV Å$^{-1}$")
plt.tight_layout()
plt.show()

## 3. Materials view: convergence by material family

In [ ]:
family_conv = (
    df_nonzero.groupby("material_family")
    .agg(rows=("Name", "size"), median_fmax=("fmax", "median"), mean_energy=("E", "mean"))
    .query("rows >= 20")
    .sort_values("rows", ascending=False)
)
family_conv["fraction_converged"] = [
    OP.converged(group, force_col="fmax", convergence_threshold=0.01).shape[0] / len(group)
    for _, group in df_nonzero.groupby("material_family")
    if len(group) >= 20
]
family_conv.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(
    family_conv["rows"],
    family_conv["median_fmax"],
    c=family_conv["fraction_converged"],
    s=90,
    cmap="viridis",
)
for label, x, y in zip(family_conv.index, family_conv["rows"], family_conv["median_fmax"], strict=False):
    ax.text(x, y, label, fontsize=8)
ax.set_title("Material-family convergence map")
ax.set_xlabel("number of rows")
ax.set_ylabel("median fmax / eV Å$^{-1}$")
fig.colorbar(scatter, ax=ax, label="fraction converged")
plt.tight_layout()
plt.show()

## 4. Lowest-energy representatives by formula

In [ ]:
formula_candidates = df_converged[df_converged["Formula"].astype(str).ne("0")].copy()
formula_minima = OP.group_min(formula_candidates, group="Formula", value="E")
formula_minima[["Name", "Formula", "record_class", "material_family", "facet", "E", "fmax"]].sort_values("E").head(30)

## 5. Lattice-parameter distributions for bulk-like rows

In [ ]:
bulk = df_converged[df_converged["record_class"].eq("bulk")].copy()
bulk[["Name", "Formula", "a", "b", "c", "gamma", "E"]].head(20)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, column in zip(axes.ravel(), ["a", "b", "c", "gamma"], strict=False):
    values = pd.to_numeric(bulk[column], errors="coerce").dropna()
    ax.hist(values, bins=25, color="#f58518", alpha=0.85)
    ax.set_title(f"Bulk distribution of {column}")
    ax.set_xlabel(column)
    ax.set_ylabel("count")
plt.tight_layout()
plt.show()

From a thesis perspective, the main conclusion is that convergence is not an
afterthought but part of the scientific interpretation. The database is large
and chemically diverse, yet most numerically meaningful rows sit in a low-fmax
regime. That makes it reasonable to proceed to adsorption chemistry.